In [ ]:
# ==========================================================================
# 업무 안내서 문장 검색 웹 화면 (Gradio) — 셀 하나로 실행
#   사용 순서: 이 셀 실행 → 셀 아래(또는 https://….gradio.live 주소)에 웹 화면이 나타남
#     ① PDF 파일 선택, 쪽 범위 입력 → [문장 만들기]
#     ② 질문 입력 → [질문하기] (또는 Enter)
#     ③ 대답(1위 문장)과 ④ 검색 결과 표(상위 5개)가 표시됨
#   끝낼 때: 셀 왼쪽의 ■(중지) 버튼
#   출처: 개인정보보호위원회, 「개인정보 보호 교육 업무 안내서」 2024.12.
# ==========================================================================

# ----- 0. 도구 설치 -----
# PIL(Pillow): Colab에 이미 설치되어 있는 이미지 도구, 지금 설치된 버전 번호를 알아내려고 불러옴
import PIL

# pip: 파이썬 도구 설치 명령, 맨 앞의 !는 터미널 명령으로 실행하라는 표시, -q: 설치 메시지 줄이기
# pillow=={PIL.__version__}: Pillow를 지금 버전 그대로 고정
#   (설치 중에 Pillow 버전이 바뀌면 이미 불러온 옛 파일과 새 파일이 섞여 ImportError가 남)
!pip install -q pdfplumber sentence-transformers gradio pillow=={PIL.__version__}

# ----- 1. 임베딩 모델 불러오기 (한 번만) -----
# SentenceTransformer: 사전학습 문장 임베딩 모델을 불러오는 클래스
# cosine_similarity: 벡터들 사이의 코사인 유사도(-1 ~ 1, 1에 가까울수록 뜻이 비슷함)를 계산하는 함수
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

# Hugging Face Hub에서 모델 파일을 내려받아 불러옴 (한 번 내려받으면 캐시에 저장됨)
model = SentenceTransformer("jhgan/ko-sroberta-multitask")

# 불러오기 완료 메시지 출력
print("모델 준비 완료")

# ----- 2. 전처리 함수: PDF → 문장 표 -----
# pdfplumber: PDF에서 글자를 뽑는 도구
# re: 정규표현식(글자 패턴) 도구
# pandas: 표(DataFrame) 다루기 도구
import pdfplumber
import re
import pandas as pd

# 문장이 시작되는 글머리 기호 패턴
#   ^ : 줄의 맨 앞
#   ⬛ ⚫ ※ • : 안내서에서 쓰는 글머리 기호
#   "- "  : 하이픈 + 띄어쓰기
#   [①-⑳] : 동그라미 숫자 ①~⑳ 중 하나
#   \d+\) : 1) 2) 처럼 숫자 + 닫는 괄호
#   [가-하]\. : 가. 나. 다. 처럼 한글 한 글자 + 마침표 + 띄어쓰기
#   \d+\. : 1. 2. 처럼 숫자 + 마침표 + 띄어쓰기
BULLET = re.compile(r"^(⬛|⚫|※|•|- |[①-⑳]|\d+\)|[가-하]\. |\d+\. )")

# 통째로 버릴 줄 패턴: 쪽 머리글('8 개인정보 보호 교육 업무 안내서')과 장 번호만 있는 줄('Ⅱ')
SKIP = re.compile(r"^\d+\s*개인정보 보호 교육 업무 안내서$|^[ⅠⅡⅢⅣⅤ]$")

# 소제목 패턴: '참고 ┃ ...' 상자 제목, '2 적용 대상' 같은 번호 소제목, '…법 제28조(…)' 같은 법조문 상자 제목
HEADING = re.compile(r"^참고\s*┃|^\d+\s+[가-힣]|제\d+조\(.+\)$")


# PDF 경로와 쪽 범위를 받아 [쪽, 문장] 표를 돌려주는 함수
def pdf_to_sentences(pdf_path, page_start, page_end):
    # 문장 단위 결과를 [쪽 번호, 원문] 형태로 모을 빈 리스트
    units = []

    # with 문: PDF 파일을 열고, 블록이 끝나면 자동으로 닫음
    with pdfplumber.open(pdf_path) as pdf:
        # 끝 쪽이 PDF 전체 쪽 수보다 크면 마지막 쪽까지만 사용
        page_end = min(page_end, len(pdf.pages))

        # 분석할 쪽 번호를 하나씩 꺼냄 (range의 끝 값은 포함되지 않으므로 +1)
        for page_no in range(page_start, page_end + 1):
            # pdf.pages는 0부터 번호가 매겨지므로 쪽 번호 - 1로 해당 쪽을 꺼냄
            page = pdf.pages[page_no - 1]

            # page.find_tables(): 이 쪽에서 표로 인식된 영역들의 목록
            # page.outside_bbox(영역): 그 영역 바깥 부분만 남긴 새 page 객체를 돌려줌 → 표 글자를 제거
            for table in page.find_tables():
                page = page.outside_bbox(table.bbox)

            # 지금 만들고 있는 문장 ([쪽 번호, 글자]), 아직 없으면 None
            cur = None

            # page.extract_text(): 이 쪽의 글자를 줄바꿈(\n)이 들어간 문자열 하나로 돌려줌
            # .splitlines(): 줄 단위 리스트로 나눔
            for line in page.extract_text().splitlines():
                # 줄 앞뒤의 공백 제거
                line = line.strip()

                # 빈 줄이거나 버릴 줄이면 다음 줄로 넘어감
                if not line or SKIP.match(line):
                    continue

                # 글머리 기호로 시작하거나 소제목이면: 지금까지 만든 문장을 저장
                if BULLET.match(line) or HEADING.match(line):
                    if cur:
                        units.append(cur)
                    # 글머리 기호면 새 문장을 시작, 소제목이면 문장을 시작하지 않음(None)
                    cur = [page_no, line] if BULLET.match(line) else None

                # 글머리 기호가 없는 줄이고 만들던 문장이 있으면: 띄어쓰기 한 칸 두고 이어 붙임
                elif cur:
                    cur[1] += " " + line

            # 쪽이 끝나면 마지막으로 만들던 문장도 저장
            if cur:
                units.append(cur)

    # [쪽 번호, 원문] 리스트를 두 열짜리 표로 변환
    df = pd.DataFrame(units, columns=["쪽", "문장"])

    # 맨 앞의 글머리 기호를 지우고(.str.replace) 앞뒤 공백 제거(.str.strip)
    df["문장"] = df["문장"].str.replace(BULLET, "", regex=True).str.strip()

    # 글자 수가 20자 이상인 문장만 남기고, 행 번호를 0부터 다시 매김
    return df[df["문장"].str.len() >= 20].reset_index(drop=True)

# ----- 3. 화면 동작 함수 -----
# 검색 결과로 보여 줄 문장 개수 (✏️ 바꿔 볼 수 있음)
TOP_K = 5


# [문장 만들기] 버튼을 누르면 실행: PDF → 문장 표 → 문장 벡터
def load_pdf(pdf_path, page_start, page_end):
    # PDF를 고르지 않았으면 안내 메시지만 돌려줌 (표와 보관함은 비움)
    if pdf_path is None:
        return "PDF 파일을 먼저 선택하세요.", None, None

    # 전처리 함수로 문장 표 만들기 (쪽 번호는 정수로 변환)
    df = pdf_to_sentences(pdf_path, int(page_start), int(page_end))

    # 문장을 하나도 찾지 못했으면 안내 메시지만 돌려줌 (글자가 없는 스캔 PDF 등)
    if len(df) == 0:
        return "문장을 찾지 못했습니다. 쪽 범위를 확인하세요.", None, None

    # 모든 문장을 벡터로 변환: (문장 수 × 768) 배열, X의 i번째 행 = df의 i번째 문장
    X = model.encode(df["문장"].tolist())

    # 처리 결과 메시지, 미리보기 표(앞 10행), 보관함에 넣을 (문장 표, 문장 벡터) 묶음을 돌려줌
    message = f"문장 {len(df)}개를 만들었습니다. 이제 질문을 입력하세요."
    return message, df.head(10), (df, X)


# [질문하기] 버튼을 누르면 실행: 질문 → 비슷한 문장 상위 TOP_K개
def ask(question, data):
    # 보관함이 비어 있으면 (PDF를 아직 처리하지 않았으면) 안내 메시지
    if data is None:
        return "먼저 PDF를 올리고 [문장 만들기]를 누르세요.", None

    # 질문이 비어 있으면 안내 메시지 (.strip(): 공백만 입력한 경우도 빈 것으로 봄)
    if not question.strip():
        return "질문을 입력하세요.", None

    # 보관함에서 문장 표와 문장 벡터를 꺼냄
    df, X = data

    # 질문을 리스트로 감싸 넣어 (1 × 768) 배열로 변환
    q = model.encode([question])

    # 질문과 모든 문장의 유사도: (1 × 문장 수) 배열 → [0]으로 1차원 배열로 꺼냄
    scores = cosine_similarity(q, X)[0]

    # argsort(): 작은 값부터의 위치 순서 → [::-1]로 뒤집어 큰 값부터 → [:TOP_K]로 앞 TOP_K개
    top = scores.argsort()[::-1][:TOP_K]

    # 해당 행들의 쪽·문장을 꺼내고, 맨 앞에 순위 열, 맨 뒤에 유사도 열(소수 셋째 자리)을 붙임
    result = df.loc[top, ["쪽", "문장"]].assign(유사도=scores[top].round(3))
    result.insert(0, "순위", range(1, len(result) + 1))

    # 대답: 1위 문장과 쪽 번호, 유사도
    best = result.iloc[0]
    answer = f"[{best['쪽']}쪽] {best['문장']}\n\n(유사도 {best['유사도']:.3f} · 안내서 원문 문장입니다. 아래 표에서 다른 후보도 확인하세요.)"

    # 대답 글과 검색 결과 표를 돌려줌
    return answer, result

# ----- 4. 웹 화면 만들고 실행 -----
# 공개 주소(….gradio.live)는 주소를 아는 누구나 접속할 수 있으므로 민감한 문서는 올리지 말 것
# gradio: 파이썬 함수로 웹 화면을 만드는 도구
import gradio as gr

# gr.Blocks(): 웹 화면 전체 틀, with 블록 안에 놓은 부품이 위에서부터 차례로 화면에 배치됨
with gr.Blocks() as demo:
    # 화면 제목과 설명 (# 은 큰 제목)
    gr.Markdown("# 📄 업무 안내서 문장 검색")
    gr.Markdown("PDF를 올리고 질문하면, 뜻이 비슷한 안내서 문장을 쪽 번호와 함께 찾아 줍니다.")

    # 보이지 않는 보관함: load_pdf가 만든 (문장 표, 문장 벡터)를 담아 ask에 넘겨줌
    data = gr.State()

    # ===== ① PDF 올리기 =====
    gr.Markdown("## ① PDF 올리기")
    # gr.Row(): 안의 부품을 가로로 나란히 배치
    with gr.Row():
        # 파일 선택 칸: .pdf 파일만 고를 수 있음, 함수에는 파일 경로(문자열)로 전달됨
        pdf_file = gr.File(label="PDF 파일", file_types=[".pdf"])
        # ✏️ 쪽 범위 입력 칸 (precision=0: 정수만), 기본값은 안내서 본문 7~17쪽
        page_start = gr.Number(value=7, label="시작 쪽", precision=0)
        page_end = gr.Number(value=17, label="끝 쪽", precision=0)
    # 버튼 (variant="primary": 강조 색 버튼)
    load_btn = gr.Button("문장 만들기", variant="primary")
    # 처리 결과 메시지 칸
    status = gr.Textbox(label="처리 결과")
    # 만들어진 문장 미리보기 표 (wrap=True: 긴 문장 줄바꿈, column_widths: 쪽·문장 열 너비)
    preview = gr.Dataframe(label="문장 미리보기 (앞 10개)", wrap=True, column_widths=["8%", "92%"])

    # ===== ② 질문하기 =====
    gr.Markdown("## ② 질문하기")
    # 질문 입력 칸 (placeholder: 칸이 비어 있을 때 흐리게 보이는 안내 글)
    question = gr.Textbox(label="질문", placeholder="예) 개인정보 보호 교육은 1년에 몇 번 해야 하나요?")
    # ✏️ 예시 질문: 누르면 질문 칸에 글자가 채워짐
    gr.Examples(
        examples=["개인정보 보호 교육은 1년에 몇 번 해야 하나요?",
                  "아르바이트나 계약직 직원도 교육을 받아야 하나요?",
                  "직원이 교육을 들었다는 증거는 무엇으로 남기나요?",
                  "교육을 공짜로 해 준다는 업체를 조심해야 하는 이유는?"],
        inputs=question,
    )
    ask_btn = gr.Button("질문하기", variant="primary")

    # ===== ③ 대답과 ④ 검색 결과 =====
    # 대답 칸 (lines=4: 4줄 높이)
    answer = gr.Textbox(label="대답 (가장 관련 있는 문장)", lines=4)
    # 검색 결과 표 (wrap=True: 긴 문장을 줄바꿈해서 표시, column_widths: 순위·쪽·문장·유사도 열 너비)
    result = gr.Dataframe(label=f"검색 결과 (상위 {TOP_K}개)", wrap=True, column_widths=["7%", "7%", "74%", "12%"])

    # [문장 만들기] 버튼 → load_pdf 실행 → 돌려준 3개 값을 status, preview, data에 순서대로 표시·저장
    load_btn.click(fn=load_pdf, inputs=[pdf_file, page_start, page_end], outputs=[status, preview, data])

    # [질문하기] 버튼 → ask 실행 → 돌려준 2개 값을 answer, result에 순서대로 표시
    ask_btn.click(fn=ask, inputs=[question, data], outputs=[answer, result])
    # 질문 칸에서 Enter를 눌러도 같은 함수 실행
    question.submit(fn=ask, inputs=[question, data], outputs=[answer, result])

# 웹 화면 실행 (debug=True: 셀이 계속 실행되며 오류 메시지를 셀 아래에 보여 줌)
demo.launch(debug=True)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 1.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.4/62.4 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 41.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.8/3.8 MB 49.0 MB/s eta 0:00:00


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/4.86k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/744 [00:00<?, ?B/s]

... (5 outputs left out)

tokenizer.json:   0%|          | 0.00/495k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/156 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

모델 준비 완료
It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://164f940e2354ff663d.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
